# 과제 2 — 당뇨병 임상시험 분석: 변수 찾기 → 표 연결 → 통계 (20점)

- **하는 일**: 동료의 SQL 6개(S1~S6)를 고칩니다. 모두 실수가 한 군데 있어 오류가 납니다.
- **분석 질환**: 제2형 당뇨병 (`'Diabetes Mellitus, Type 2'`), 모든 학생 공통
- **제출**: 13주차 수업 후 1주일 이내, LMS 에 `과제2.ipynb`

| 배점 | 점수 | 기준 |
|---|---|---|
| 단계별 SQL | 8점 | S1~S6 ✅ 1점씩 + 실행 · 형식 2점 |
| 오류 설명 | 5점 | ✏️ S1~S6 한 줄씩 (6칸 중 5칸이면 만점) |
| 결과표 | 4점 | ✏️ 결과표 |
| 해석 | 3점 | ✏️ 해석 |

## 0. 시작하기 전에

- Colab 에서는 먼저 **`File → Save a copy in Drive`** 를 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` 같은 줄은 지우지 마세요.
- 문항 SQL 을 실행하면 결과 아래에 **✅ / ❌** 가 바로 나옵니다.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.
- ✏️ 칸은 첫 줄을 두고 `→` 뒤에 적습니다.

### 고치는 방법

1. 동료의 SQL 을 실행해 `Error:` 줄을 읽고, 💡 힌트를 보고 한 군데만 고칩니다.
2. 고친 SQL 을 실행해 결과 아래에 ✅ 가 나오면 1점입니다.
3. 무엇을 왜 고쳤는지 ✏️ 칸에 한 줄로 적습니다. 예) `→ brief_titel 을 brief_title 로 고침 — 열 이름 오타`

In [ ]:
#@title [1번] ✏️ 학번 · 이름을 입력하고 실행하세요 (준비 셀) { display-mode: "form" }
학번 = ""  #@param {type:"string"}
이름 = ""  #@param {type:"string"}
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            if parts and _re.match(r"^\[\d+번\]$", parts[0]):      # 수업용 셀 번호 [12번] 은 건너뜀
                parts = parts[1:]
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)
    if tag and tag in globals().get("ANSWERS", {}):         # 문항 셀(-- Q1)이면 실행하자마자 ✅/❌ 표시
        check(tag, getattr(result, "result", None))

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "SQL 에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, "결과가 표가 아닙니다. SQL 이 SELECT 로 끝나는지 확인하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 실행 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

MESH = ["Diabetes Mellitus, Type 2"]
학번, 이름 = str(globals().get("학번", "")).strip(), str(globals().get("이름", "")).strip()
MY_MESH = MESH[int(_hl.md5(학번.encode()).hexdigest(), 16) % len(MESH)]      # 지금은 모든 학생 공통
if not 학번 or 학번 == "20260000":
    print("⚠️ 학번이 비어 있습니다. 위 칸에 학번과 이름을 입력하고 이 준비 셀을 다시 실행하세요.")
else:
    print(f"연결 완료.  {이름}({학번})")
print(f"분석 질환: {MY_MESH} (제2형 당뇨병, 모든 학생 공통)")

def 제출확인():
    """제출 전에 실행: 문항별 결과와 제출 방법을 보여 줍니다."""
    print(f"제출자: {이름 or '(이름 없음)'} ({학번 or '학번 없음'})  /  분석 질환: {MY_MESH}")
    진행확인()
    print("\n제출: ① Runtime → Run all (오류에서 멈추면 아래 셀을 누르고 Runtime → Run after)  ② File → Download → Download .ipynb  ③ LMS 에 과제2.ipynb 제출")

ANSWERS = _load_answers("eNql0j1Lw0AYwPGvUm526L0m59foKB3uVQOhkSTSoRQcOggKTtIK2VxchCL4pdL0O3inFB5DCtJOufyPCz+e3AJNMLocLVCVXYcn4lxwopzC0nJ0MULFkV4W8yp0HJamyOPyCnWPzX696R42aBryTTar48mfr5TWlc6GV6/yysVyW2fFTOWHtAxtQqBEU6lSmkjNdAolnhmnE5pyoz2QcChpvz7a7X3c/Y+pLu+OkigkMcINwzq1UnlI6vfB4byudm9Pv5xT58MgRiglUya80IxATL8PYfYvTRjReRgOMePEcMswYY4yiOn3IUz7GcfSNe+nSgSUWKqxUwYTLimU9PtBMv7zk9bb3fPq3Huz/AbwYwlP")   # 자동 확인용 (정답 SQL 은 없음)

## 데이터 구성 — 어떤 표가 있고, 어떻게 연결되나

표 이름과 연결 키(FK)는 이 그림과 표에서 확인합니다.

![aact_mini 표 구조 한눈에 보기](https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini_erd.png)

| 표 | 한 행이 뜻하는 것 | 기본키 (PK) | 외래키 (FK) | studies 와의 관계 | 주로 쓰는 열 |
|---|---|---|---|---|---|
| `studies` | 임상시험 1건 | `nct_id` | — | 기준이 되는 표 | brief_title, study_type, overall_status, phase, start_date, enrollment, results_first_posted_date |
| `eligibilities` | 선정 · 제외 기준 | `id` | `nct_id` → studies | 1 : 1 (시험마다 1행) | gender, minimum_age (글자) |
| `sponsors` | 스폰서 1곳 | `id` | `nct_id` → studies | 1 : N (하나 이상) | agency_class, lead_or_collaborator, name |
| `conditions` | 대상 질환 1개 (연구자가 적은 이름) | `id` | `nct_id` → studies | 1 : N (하나 이상) | name (자유 입력이라 표기가 제각각) |
| `browse_conditions` | 표준 질환명(MeSH) 1개 | `id` | `nct_id` → studies | 1 : N (없을 수도 있음) | mesh_term |
| `countries` | 참여 국가 1곳 | `id` | `nct_id` → studies | 1 : N (없을 수도 있음) | name, removed (TRUE 면 등록 후 빠진 국가) |
| `facilities` | 수행 기관 1곳 | `id` | `nct_id` → studies | 1 : N (없을 수도 있음) | name, country |

**읽는 법** (5 · 6주차)

1. **기본키** — 한 행을 하나로 찾는 열입니다. `studies` 는 `nct_id`(임상시험 등록번호), 나머지 표는 `id` 입니다.
2. **외래키** — 다른 표의 `nct_id` 는 `studies.nct_id` 를 가리킵니다. 그래서 표를 연결(JOIN)할 때는 늘 `ON studies.nct_id = 다른표.nct_id` 로 잇습니다.
3. **1 : N** — 시험 1건에 국가 · 기관 · 질환이 여러 개면 그 표에는 행이 여러 개입니다. 연결하면 시험 1건이 여러 행이 되므로, 시험 수는 `COUNT(DISTINCT studies.nct_id)` 로 셉니다. 또 `browse_conditions` · `countries` · `facilities` 에는 행이 하나도 없는 시험도 있어서, INNER JOIN 하면 그 시험이 빠집니다.

In [ ]:
%%sql
-- [2번] 확인: studies 에서는 1행인 시험이 countries 와 연결하면 국가 수만큼 여러 행이 됩니다
SELECT studies.nct_id, overall_status, countries.name AS 국가
FROM studies
JOIN countries
  ON studies.nct_id = countries.nct_id
WHERE studies.nct_id = 'NCT01795521'

## 1. 변수 찾기 — 데이터 사전 (11주차)

분석 전에 데이터 사전에서 표 · 열 이름 · 타입을 확인합니다 (11주차 분석 업무 흐름 ③).

In [ ]:
%%sql
-- [3번] [예시] 데이터 사전 보기 — 'browse_conditions' 를 다른 표 이름으로 바꿔도 됩니다
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'browse_conditions'
ORDER BY ordinal_position;

### S1 · 당뇨병 시험 수

동료가 원한 것: 제2형 당뇨병 시험이 몇 건인지 세어 주세요.

💡 힌트: 열 이름은 데이터 사전에서 확인합니다 (11주차 데이터 사전).

In [ ]:
%%sql
-- [4번] S1
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(DISTINCT nct_id) AS 시험수
FROM browse_conditions
WHERE mesh = 'Diabetes Mellitus, Type 2';

✏️ **S1 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

### S2 · 참여 국가 상위 5곳

동료가 원한 것: 전체 자료에서 시험이 가장 많은 참여 국가 5곳을 보여 주세요 (빠진 국가 removed 는 제외).

💡 힌트: 표 이름은 위 데이터 구성 그림에서 확인합니다 (7주차 데이터 구성).

In [ ]:
%%sql
-- [5번] S2
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT name AS 국가,
       COUNT(DISTINCT nct_id) AS 시험수
FROM country
WHERE removed = false
GROUP BY name
ORDER BY 시험수 DESC, 국가
LIMIT 5;

✏️ **S2 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

## 2. 표 연결 — JOIN (8주차 · 11주차 연결 키)

두 표를 잇는 조건은 ON 절에 적습니다.

### S3 · 완료된 당뇨병 시험 수

동료가 원한 것: 완료(COMPLETED)된 제2형 당뇨병 시험이 몇 건인지 세어 주세요.

💡 힌트: 두 표를 잇는 연결 키(FK)는 무엇인가요? (11주차 연결 키 · 8주차 JOIN)

In [ ]:
%%sql
-- [6번] S3
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(DISTINCT studies.nct_id) AS 완료시험
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.id
WHERE mesh_term = 'Diabetes Mellitus, Type 2'
  AND overall_status = 'COMPLETED';

✏️ **S3 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

### S4 · 한국에서 진행한 당뇨병 시험 수

동료가 원한 것: 한국(South Korea)에서 진행한 제2형 당뇨병 시험이 몇 건인지 세어 주세요 (빠진 국가는 제외).

💡 힌트: ON 에는 이미 연결한 표만 쓸 수 있습니다 (8주차 JOIN).

In [ ]:
%%sql
-- [7번] S4
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(DISTINCT studies.nct_id) AS 한국시험
FROM studies
JOIN countries
  ON countries.nct_id = browse_conditions.nct_id
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
WHERE mesh_term = 'Diabetes Mellitus, Type 2'
  AND countries.name = 'South Korea'
  AND removed = false;

✏️ **S4 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

## 3. 통계 — CASE WHEN · 비율 · GROUP BY (8 · 9주차)

비율(%) = 분자 ÷ 분모 × 100. 100.0 을 곱해야 소수점이 남습니다.

### S5 · 결과 게시율 (%)

동료가 원한 것: 완료된 제2형 당뇨병 시험 중 결과가 게시된 비율(%)을 구해 주세요.

💡 힌트: CASE WHEN 은 무엇으로 끝나나요? (8주차 CASE WHEN)

In [ ]:
%%sql
-- [8번] S5
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT ROUND(100.0 * COUNT(CASE WHEN results_first_posted_date IS NOT NULL
                                THEN 1)
             / COUNT(*), 1) AS 게시율
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
WHERE mesh_term = 'Diabetes Mellitus, Type 2'
  AND overall_status = 'COMPLETED';

✏️ **S5 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

### S6 · 연도별 당뇨병 시험 수

동료가 원한 것: 제2형 당뇨병 시험을 시작 연도별로 세어 주세요.

💡 힌트: YEAR() 에는 날짜(DATE) 타입 열만 넣을 수 있습니다. 열의 타입은 데이터 사전에서 확인하세요 (11주차 데이터 사전).

In [ ]:
%%sql
-- [9번] S6
-- [고칠 SQL] 동료가 쓴 SQL — 실수 한 군데를 고쳐 실행하세요
SELECT YEAR(start_month_year) AS 연도,
       COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
WHERE mesh_term = 'Diabetes Mellitus, Type 2'
GROUP BY 연도
ORDER BY 연도;

✏️ **S6 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

## 4. 결과표와 해석

고친 SQL 의 결과로 채웁니다.

✏️ **결과표** (두 번 눌러 빈칸에 적으세요):

| 항목 | 단계 | 값 |
|---|---|---|
| 당뇨병 시험 수 | S1 |  |
| 완료 시험 | S3 |  |
| 결과 게시율 (%) | S5 |  |
| 한국 시험 | S4 |  |
| 시험이 가장 많은 해 | S6 |  |
| 참여 1위 국가 (전체 자료) | S2 |  |

말할 수 있는 것(조건 + 숫자)과 말할 수 없는 것(이유와 함께)을 나누어 씁니다 (13주차 결과 해석).

✏️ **해석** (두 번 눌러 → 뒤에 3~5문장):
→ 

## 5. 제출 확인

In [ ]:
제출확인()   # [10번]

**제출** — 모든 문항이 ✅ 인 상태로 저장(`File → Download → Download .ipynb`)한 뒤 **LMS** 에 제출하세요.